# ShipwakeAIS — Run Pipeline

Edit the paths in **Cell 1** before running.  
Each cell is independent — re-run any stage without restarting.


In [ ]:
# === EDIT THESE PATHS FOR YOUR RUN ===
ais_csv       = r"examples/ais/AIS_2563.csv"
coastline_shp = r"examples/coastline/Coast_P1.shp"
bathy_file    = r"examples/bathymetry/61803960_WestCoast_HD_25m_mCD_Prod_v20260220.mesh"
tide_dfs0     = r"examples/tide/Predicted Water Level (CD)_2024_WestCoast_6min.dfs0"
output_dir    = r"output/"

# === PARAMETERS (adjust as needed) ===
cb_method      = "L_Le"   # 'L_Le', 'B_Le', or 'table'
gravity        = 9.78     # Singapore local gravity
max_prop_m     = 2000.0   # maximum wake propagation distance (m)
wake_cutoff_m  = 0.01     # minimum shore wave height to record (m)

In [ ]:
# === Cell 2: Build config ===
from aiswakepy.config import load_config

config = load_config({
    "ais": {"raw_csv": ais_csv},
    "vessel": {"cb_method": cb_method},
    "bathymetry": {"source": bathy_file, "tide_dfs0": tide_dfs0},
    "coastline": {"shapefile": coastline_shp},
    "wave": {"gravity": gravity},
    "impact": {"max_propagation_m": max_prop_m, "wake_cutoff_m": wake_cutoff_m},
    "output": {"directory": output_dir},
})
print("Config loaded OK")


In [ ]:
# === Cell 3: Stage 1 — AIS Filter ===
from aiswakepy.stages.filter import filter_ais

df_filtered = filter_ais(
    csv_path=config.ais.raw_csv,
    coastline_shp=config.coastline.shapefile,
    gap_s=config.ais.traj_gap_s,
    spacing_m=config.ais.interp_spacing_m,
    trigger_m=config.ais.interp_trigger_m,
)
print(f"Filtered: {len(df_filtered)} rows")
df_filtered.head()


In [ ]:
# === Cell 4: Stage 2 — Depth + Tide Assignment ===
from aiswakepy.stages.depth import assign_depth

df_depth = assign_depth(
    df=df_filtered,
    bathy_path=config.bathymetry.source,
    tide_dfs0_path=config.bathymetry.tide_dfs0,
    underkeel_margin_m=config.bathymetry.underkeel_margin_m,
)
print(f"After depth filter: {len(df_depth)} rows")
df_depth[["mmsi", "longitude", "latitude", "WaterDepth", "draught"]].describe()


In [ ]:
# === Cell 5: Stage 3 — Wave Parameters ===
from aiswakepy.stages.wave_params import compute_wave_params, export_gis

df_wave = compute_wave_params(
    df=df_depth,
    cb_method=config.vessel.cb_method,
    g=config.wave.gravity,
    rho=config.wave.rho_water,
    min_froude_m=config.wave.min_froude_m,
    max_froude_m=config.wave.max_froude_m,
    max_bf=config.wave.max_bf,
    max_sog_knots=config.wave.max_sog_knots,
    max_bl_ratio=config.wave.max_bl_ratio,
)
print(f"Wave events: {len(df_wave)}")
df_wave[["mmsi", "FroudeM", "H_Kriebel", "Theta", "WakeDirPort", "WakeDirStarboard"]].describe()

In [ ]:
# === Cell 6: Stage 4 — Shore Impact ===
from aiswakepy.stages.shore_impact import compute_shore_impact

df_impact = compute_shore_impact(
    df_wave=df_wave,
    coastline_shp=config.coastline.shapefile,
    max_propagation_m=config.impact.max_propagation_m,
    wake_cutoff_m=config.impact.wake_cutoff_m,
    g=config.wave.gravity,
)
print(f"Shore impact events: {len(df_impact)}")
df_impact.head()


In [ ]:
# === Cell 7: Visualisation ===
from pathlib import Path
from aiswakepy.viz.wave_map import plot_wave_height_map, plot_wave_period_map

out = Path(output_dir)
out.mkdir(parents=True, exist_ok=True)

plot_wave_height_map(df_impact, coastline_shp, out / "WaveHeightMap.png")
plot_wave_period_map(df_impact, coastline_shp, out / "WavePeriodMap.png")
print("Maps saved")


In [ ]:
# === Cell 8: Export results ===
from pathlib import Path

out = Path(output_dir)
df_impact.to_csv(out / "shore_impact.csv", index=False)
export_gis(df_wave).to_csv(out / "wave_params_gis.csv", index=False)
df_wave.to_parquet(out / "wave_params.parquet", index=False)
print(f"Results saved to {out.resolve()}")
